In [1]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False)

result = ocr.predict(
    input="../img_test/OCR/manuscrito.jpg")


"""
Nota: al parecer PaddleOCR soporta la entrada paralela de imágenes, ofreciendo tambien resultados paralelos.
Lo de abajo pertenece al código de ejemplo de PaddleOCR. Este ofrece una carpeta  OUTPUT que contiene imágenes correspondientes a los cuadros delimitadores de los textos detectados, así como un archivo JSON con los resultados de la detección.
"""
for res in result:
    #res.print()
    res.save_to_img("output")
    res.save_to_json("output")


Creating model: ('PP-OCRv5_mobile_det', None)
Using official model (PP-OCRv5_mobile_det), the model files will be automatically downloaded and saved in C:\Users\Sam\.paddlex\official_models.
c:\Users\Sam\Desktop\project_ia_face\faceenv\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_mobile_rec', None)
Using official model (PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in C:\Users\Sam\.paddlex\official_models.


In [3]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False)

result = ocr.predict(
    input="../img_test/OCR/texto.jpg")


contenido = result[0]["rec_texts"]
contenido

Creating model: ('PP-OCRv5_mobile_det', None)
Using official model (PP-OCRv5_mobile_det), the model files will be automatically downloaded and saved in C:\Users\Sam\.paddlex\official_models.
Creating model: ('PP-OCRv5_mobile_rec', None)
Using official model (PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in C:\Users\Sam\.paddlex\official_models.


['La gallina Josefina puso un huevo en',
 'la cocina. Carolina lo encontro y wna',
 'tortilla preparo. Sus hijos, Pedro y Anita,',
 'se la comieron enterita. Su padre, el Sr.',
 'Manuel, tortilla no quiso comer, asi que',
 'en un momento, y tan feliz y',
 'comtento, se preparo con mucho arte.',
 'wm poco de pan con tomate.',
 'La abuela de Pedro y Anita, se llama',
 'Susamita, la llamam asi porque aumque',
 'es muuy mayor, siempre ha sido muy',
 'bajita. Tiene el pelo ny blanco, y los',
 'ojos muny azules, le gusta mucho coser, y',
 'los pasteles y dulces.',
 'Su abuelo se llama Ramón, y de joven',
 'fué carpintero, ahora que está jubilado.',
 'en su huerto, planta lechugas y',
 'rálamos.']

In [75]:
import re

def extraer_datos_ocr(lista_texto):
    datos = {
        "codigo": "",
        "dni": "",
        "apellidos": "",
        "nombres": "",
        "facultad": "",
        "carrera": "",
        "expira": "",
    }

    patron_codigo = re.compile(r'^\d{10}$')
    patron_dni = re.compile(r'^\d{8}$')
    
    expira = []  # Lista para almacenar partes de la fecha de expiración
   
    #extracción por iteración de elementos de la lista de texto
    for i, elemento in enumerate(lista_texto):
        elemento = elemento.lower().replace(":", "").strip() #<-- normalizamos cualquier entrada

        if patron_codigo.match(elemento):
            datos["codigo"] = elemento
        elif patron_dni.match(elemento):
            datos["dni"] = elemento
        elif re.fullmatch(r'\d{2}', elemento):
            expira.append(elemento)  # Asumimos que es parte de la fecha de expiración

    if expira:
        datos["expira"] = "-".join(expira)  # Unimos las partes de la fecha de expiración
    
    #extracción por coincidencia de patrones en texto unificado
    texto = "\n".join(contenido)
    #------APELLIDOS
    pat = re.compile(
        r'(?i)apellidos:?\s*'        # “Apellidos:” (case-insensitive)
        r'([A-ZÁÉÍÓÚÑÜ]+)'           # 1er apellido (solo letras mayúsculas/acentos)
        r'(?:\s+\d+)*'               # opcionalmente saltar bloques de números
        r'\s+([A-ZÁÉÍÓÚÑÜ]+)'        # 2º apellido
    )
    m = pat.search(texto)
    if m:
        datos["apellidos"] = f"{m.group(1)} {m.group(2)}"

    #------NOMBRES
    pat_nombres = re.compile(
        r'(?i)nombres:?\s*'            # “Nombres:” case-insensitive
        r'([^\n]+)'            # captura mayúsculas y espacios
    )
    m_nombres = pat_nombres.search(texto)
    if m_nombres:
        datos["nombres"] = m_nombres.group(1).strip()


    #------FACULTAD
    pat_facultad = re.compile(
        r'(?i)facultad:?\s*'        # “Facultad:” (case-insensitive)
        r'([^\n]+)'         # captura mayúsculas y espacios
    )
    m_fac = pat_facultad.search(texto)
    if m_fac:
        datos["facultad"] = m_fac.group(1).strip()

    #------CARRERA
    pat_carrera = re.compile(
        r'(?i)carrera:?\s*\n*\s*'    # “Carrera:” con o sin “:” y posibles saltos de línea
        r'([A-ZÁÉÍÓÚÑÜ\.\s]+)'       # captura letras, puntos y espacios
    )
    m_car = pat_carrera.search(texto)
    if m_car:
        datos["carrera"] = m_car.group(1).strip()

    return datos

extraer_datos_ocr(contenido)

{'codigo': '2020102322',
 'dni': '71469654',
 'apellidos': 'PINEDO AQUINO',
 'nombres': 'MELANY ALEXANDRA',
 'facultad': 'INGENIERIA',
 'carrera': 'ING.DE SISTEMAS',
 'expira': '08-10-25'}